In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit.models.segmentation import segment
from fundus_toolkits import FundusData
from fundus_vessels_toolkit import VTree
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.pipelines.avseg_to_tree import GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TreeTopology, optimal_lines
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

## Load Image and Segment AV, OD, Macula


In [3]:
PATH = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/MAPLES-DR/")
RAW = PATH / "1-images"
AV = PATH / "2-av"
TOPO = PATH / "3-topo"
IMG = sorted(list(AV.glob("*.png")))[5].stem  # 18

fundus_gt = FundusData(image=RAW / (IMG + ".png"), av=AV / (IMG + ".png"))
trees_gt = VTree.load(TOPO / f"{IMG}_art.npz"), VTree.load(TOPO / f"{IMG}_vei.npz")
topo_gt = (
    TreeTopology.from_tree(trees_gt[0], expand_labels_by=5),
    TreeTopology.from_tree(trees_gt[1], expand_labels_by=5),
)


od_mac = segment(open_image(RAW / (IMG + ".png"))).numpy(force=True).argmax(axis=0)
fundus_gt = fundus_gt.update(od=od_mac == 1, macula=od_mac == 2, reshape_method="resize")
fundus = fundus_gt.copy()
_ = segment_av(fundus)

av2tree = GNNAVSegToTree()
graph = av2tree.to_vgraph(fundus).sort_branches_by_nodesID()

print(IMG)

20051020_64007_0100_PP


In [4]:
digraph = VBranchDigraph.from_graph(graph, max_distance=200)
digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])
solved_tree = digraph.optimize_tree(keep_invalid_branch=True)

m = Mosaic(
    3, cols_titles=["Predicted", "Predicted with GT Topology", "Ground Truth"], cell_height=600, background=fundus.image
)
fundus.draw(view=m[0])
draw_graph(digraph.graph, view=m[0], edge_labels=True, node_labels=True)
fundus.draw(view=m[1])
draw_tree(
    solved_tree,
    view=m[1],
    branch_color="subtree",
    bspline_dir=True,
)
fundus_gt.draw(view=m[2])
# m[2].add_image(
#     np.stack(
#         [
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             topo_gt[1].fuzzy_skeleton_map,
#         ]
#     ).transpose(1, 2, 0),
#     name="skeleton",
#     opacity=0.9,
# )
# m[2].add_image(
#     np.stack(
#         [
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             topo_gt[1].rank_map,
#         ]
#     ).transpose(1, 2, 0),
#     name="skeleton",
#     opacity=0.9,
# )
draw_trees(trees_gt, view=m[2])  # , edge="skeleton")
m

[ WARN:0@8.972] global loadsave.cpp:1617 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [5]:
VBranchDigraph.from_graph(graph, max_distance=200)

In [6]:
pd.DataFrame(digraph.lines_by_branch(199)).head(50)

,0,1,2,3,4,5,6
0,209.0,209.0,199.0,209.0,1.0,0.982730,0.992682
1,279.0,283.0,199.0,209.0,0.0,0.997190,0.992682
2,199.0,127.0,114.0,265.0,0.0,0.992682,0.997162
3,199.0,127.0,67.0,52.0,0.0,0.992682,0.997047
4,199.0,127.0,261.0,36.0,0.0,0.992682,0.996970
5,199.0,127.0,77.0,249.0,0.0,0.992682,0.996320
6,199.0,127.0,104.0,12.0,0.0,0.992682,0.995977
7,199.0,127.0,9.0,12.0,0.0,0.992682,0.995696
8,280.0,284.0,199.0,209.0,0.0,0.995239,0.992682
9,199.0,127.0,68.0,52.0,0.0,0.992682,0.995102


In [7]:
solved_tree.branch_tree[249]

np.int64(311)

In [ ]:
VBranchDigraph.from_graph(graph, max_distance=300)
digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])

In [ ]:
from fundus_vessels_toolkit.segment_to_graph.geometry_parsing import derive_tips_geometry_from_curve_geometry
from fundus_vessels_toolkit.segment_to_graph.graph_simplification import find_facing_tips
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import prepare_graph_for_reconnections

max_distance = 200
max_angle = 30
tan_max_angle = 80
pos_tolerance = 25
_, candidates = prepare_graph_for_reconnections(
    graph, max_distance=max_distance, max_angle=max_angle, av_attr="av", inplace=True
)
derive_tips_geometry_from_curve_geometry(graph, tangent=True, inplace=True)

facing_tips = find_facing_tips(
    graph,
    max_distance=max_distance,
    max_angle=max_angle,
    tan_max_angle=tan_max_angle,
    pos_tolerance=pos_tolerance,
    as_mask=False,
)
for b0, tip0, n1 in candidates:
    node = graph.node(n1)
    b1 = np.array(node.adjacent_branch_ids)
    tip1 = np.where(node.adjacent_branches_first_node, 0, 1)
    facing_tips[b0, tip0, b1, tip1] = True
    facing_tips[b1, tip1, b0, tip0] = True
facing_tips |= graph.branch_tips_connectivity_matrix()

line_list = np.argwhere(facing_tips)

B = graph.branch_count
line_list = [
    line_list,
    np.stack([np.full(B, -1), np.zeros(B), np.arange(B), np.zeros(B)], axis=-1).astype(np.int_),
    np.stack([np.full(B, -1), np.zeros(B), np.arange(B), np.ones(B)], axis=-1).astype(np.int_),
]
line_list = np.vstack(line_list)

In [ ]:
%timeit VBranchDigraph.from_graph(graph, max_distance=300)

212 ms ± 8.25 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%timeit digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])

40.2 ms ± 221 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [16]:
def draw_cone(branch_id: int, first_tip: bool, view=None, pos_tolerance=15, max_dist=100, max_angle=30):
    sqr_max_dist = max_dist * max_dist
    sqr_pos_tolerance = pos_tolerance * pos_tolerance

    min_cos = np.cos(np.deg2rad(max_angle))

    geodata = graph.geometric_data()
    tips_pos = (
        geodata.tip_coord(branch_id=branch_id, first_tip=first_tip).astype(np.float64).reshape(-1, 2)
    )  # [branch_id x (tip0, tip1), (y,x)]
    tips_tan = geodata.tip_tangent(branch_id=branch_id, first_tip=first_tip).reshape(
        -1, 2
    )  # [branch_id x (tip0, tip1), (y,x)]

    yy, xx = np.meshgrid(np.arange(fundus.image.shape[1]), np.arange(fundus.image.shape[2]), indexing="ij")
    yx = np.stack((yy, xx), axis=-1).reshape(-1, 2)

    tips_dtan = tips_pos[:, None, :] - yx[None, :, :]  # (tip_origin, tip_destination, yx)
    tips_dsqr = np.square(tips_dtan).sum(axis=2)
    tips_dtan /= np.sqrt(tips_dsqr)[..., None] + 1e-8

    # === VICINITY CHECK ===
    # Given a tip p0 with tangent t0 (oriented towards its curve)
    # we define a cone oriented towards -t0 with apex at p0 + t0 * pos_tolerance (so the tip itself is inside the cone)
    # and opening angle max_angle at distance pos_tolerance and 60 degrees at distance 0 from the apex.
    apex = tips_pos + tips_tan * pos_tolerance  # Cone apex position
    apex2tips = apex[:, None, :] - yx[None, :, :]
    apex2tips_dsqr = np.square(apex2tips).sum(axis=2)
    apex2tips /= np.sqrt(apex2tips_dsqr)[..., None] + 1e-8
    apex_cos = (tips_tan[:, None, :] * apex2tips).sum(axis=2)
    inside_cone = ((apex_cos >= min_cos) | (tips_dsqr <= sqr_pos_tolerance)) & (tips_dsqr <= sqr_max_dist)
    yx = yx[inside_cone[0]]
    map = np.zeros(fundus.image.shape[1:], dtype=np.uint8)
    map[yx[:, 0], yx[:, 1]] = 1
    if view is not None:
        view.add_label(map, "cone", opacity=0.2)